<a href="https://colab.research.google.com/github/AdvayRoongta/audio-initial-dataset/blob/main/scripts/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiments: audio frames -> LLM tokens

How much of Qwen2-Audio's input does one second of audio actually take up?

The audio encoder downsamples in fixed steps, so clip duration maps to a
predictable number of token positions. This notebook measures that mapping,
which we need before any patching / attention work -- otherwise we don't know
*which* positions in the sequence are the audio.

Sections 1-4 only need the **processor** (no GPU, no 17 GB download).
Section 5 loads the full model and needs a GPU.

Run `load_datasets.ipynb` first so the clips exist in Drive.

## Setup

In [ ]:
import os

# Get to the repo root: clone it on a fresh Colab, otherwise we're already there.
if os.path.exists("requirements.txt"):
    pass
elif os.path.exists("../requirements.txt"):
    %cd ..
else:
    !git clone -q https://github.com/AdvayRoongta/audio-initial-dataset.git repo
    %cd repo

!pip install -q -r requirements.txt


## Point at the cached clips

Same Drive folder `load_datasets.ipynb` wrote to. Nothing is downloaded here.

In [ ]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd()

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/audio-initial-dataset")
except ImportError:
    DATA_ROOT = REPO_ROOT

with open("examples.json") as f:
    examples = json.load(f)

# Paths in examples.json are relative to DATA_ROOT.
missing = [e["id"] for e in examples if not (DATA_ROOT / e["audio_path"]).exists()]
print(f"{len(examples)} examples, {len(missing)} missing audio")
if missing:
    print("run load_datasets.ipynb first ->", missing[:5])


## 1. The downsampling chain

Audio goes through fixed-size steps before the language model sees it:

```
waveform          16 kHz
  |  mel spectrogram, hop=160            -> 100 frames/sec
  |  conv stride 2                       ->  50 frames/sec
  |  average pool stride 2               ->  25 tokens/sec
  v
audio tokens      25/sec = 40 ms each, then projected to the LM's width
```

So **duration sets the number of token positions; the hidden width never
changes**. Let's confirm those constants from the model config rather than
trusting the diagram.

In [ ]:
from transformers import AutoProcessor, Qwen2AudioConfig

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
config = Qwen2AudioConfig.from_pretrained(MODEL_ID)

fe = processor.feature_extractor
SR = fe.sampling_rate
FRAMES_PER_SEC = SR / fe.hop_length          # mel frames per second
TOKENS_PER_SEC = FRAMES_PER_SEC / 4          # two stride-2 steps
SEC_PER_TOKEN = 1 / TOKENS_PER_SEC

print(f"sampling rate     {SR}")
print(f"mel frames/sec    {FRAMES_PER_SEC:.0f}")
print(f"audio tokens/sec  {TOKENS_PER_SEC:.0f}   ({SEC_PER_TOKEN*1000:.0f} ms per token)")
print(f"audio encoder dim {config.audio_config.d_model}")
print(f"LM hidden dim     {config.text_config.hidden_size}")
print(f"audio token id    {config.audio_token_index}")


## 2. Duration -> number of tokens

The two stride-2 steps use integer arithmetic, so the exact count is:

```
n = ((mel_frames - 1) // 2 + 1 - 2) // 2 + 1
```

`n_audio_tokens` is what we compare against the processor's real output.

In [ ]:
def n_mel_frames(duration_sec):
    """Mel frames the feature extractor produces for a clip this long."""
    return int(duration_sec * FRAMES_PER_SEC)


def n_audio_tokens(mel_frames):
    """Mel frames -> audio token positions (conv stride 2, then pool stride 2)."""
    after_conv = (mel_frames - 1) // 2 + 1
    return (after_conv - 2) // 2 + 1


for d in [1, 2.5, 5, 10, 30]:
    print(f"{d:>5} s -> {n_mel_frames(d):>5} mel frames -> {n_audio_tokens(n_mel_frames(d)):>4} tokens")


## 3. Check it against the real processor

The feature extractor always pads to 30 s, so `input_features` is the same
shape for every clip. The number of *audio tokens* is not — it comes from
`feature_attention_mask`. Confirm that, and that our formula matches.

In [ ]:
import inspect

import librosa

AUDIO_TOKEN_ID = config.audio_token_index

# transformers renamed this kwarg (audios= -> audio=) and silently ignores the
# wrong one, so pick whichever this version actually takes.
AUDIO_KWARG = "audio" if "audio" in inspect.signature(processor.__call__).parameters else "audios"


def encode(example):
    """Run one example through the processor. Returns inputs + duration."""
    path = DATA_ROOT / example["audio_path"]
    audio, _ = librosa.load(path, sr=SR)

    conversation = [{"role": "user", "content": [
        {"type": "audio", "audio_url": str(path)},
        {"type": "text", "text": example["text_prompt"]},
    ]}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[text], sampling_rate=SR, return_tensors="pt",
                       **{AUDIO_KWARG: [audio]})
    return inputs, len(audio) / SR


ex = examples[0]
inputs, duration = encode(ex)

valid_frames = int(inputs["feature_attention_mask"].sum())
actual = int((inputs["input_ids"] == AUDIO_TOKEN_ID).sum())
predicted = n_audio_tokens(valid_frames)

print(f"clip                 {ex['id']}  ({duration:.2f} s)")
print(f"input_features       {tuple(inputs['input_features'].shape)}   <- padded to 30 s")
print(f"valid mel frames     {valid_frames}")
print(f"audio tokens         {actual}  (formula predicted {predicted})")
print(f"total prompt tokens  {inputs['input_ids'].shape[-1]}")
assert actual == predicted, "formula disagrees with the processor"


Now sweep every example to confirm the relationship is linear.

In [ ]:
rows = []
for e in examples:
    if not (DATA_ROOT / e["audio_path"]).exists():
        continue
    inputs, duration = encode(e)
    rows.append({
        "id": e["id"],
        "conflict_type": e["conflict_type"],
        "duration_sec": round(duration, 2),
        "audio_tokens": int((inputs["input_ids"] == AUDIO_TOKEN_ID).sum()),
        "total_tokens": inputs["input_ids"].shape[-1],
    })

for r in rows[:10]:
    print(f"{r['id']:<16} {r['duration_sec']:>5.2f} s  {r['audio_tokens']:>4} audio  "
          f"{r['total_tokens']:>4} total  ({r['audio_tokens']/r['duration_sec']:.1f} tok/s)")
print(f"\n{len(rows)} examples encoded")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter([r["duration_sec"] for r in rows], [r["audio_tokens"] for r in rows], s=18)
plt.xlabel("clip duration (s)")
plt.ylabel("audio tokens")
plt.title(f"~{TOKENS_PER_SEC:.0f} audio tokens per second")
plt.grid(alpha=0.3)
plt.show()


## 4. Which token covers which moment

Each audio token is one 40 ms slice, in order. That gives us both directions:
token index -> time span, and timestamp -> token index.

In [ ]:
def token_to_time(k):
    """Audio token k -> (start, end) in seconds."""
    return k / TOKENS_PER_SEC, (k + 1) / TOKENS_PER_SEC


def time_to_token(t):
    """Timestamp in seconds -> audio token index."""
    return int(t * TOKENS_PER_SEC + 1e-9)  # epsilon: floats land just under bin edges


def audio_token_positions(inputs):
    """Where the audio sits in the prompt: (first, last) index into input_ids."""
    pos = (inputs["input_ids"][0] == AUDIO_TOKEN_ID).nonzero().flatten()
    return int(pos[0]), int(pos[-1])


inputs, duration = encode(examples[0])
start, end = audio_token_positions(inputs)

print(f"audio occupies input_ids[{start}:{end+1}]  ({end - start + 1} positions)")
print(f"token 0   covers {token_to_time(0)[0]:.2f}-{token_to_time(0)[1]:.2f} s")
print(f"token 10  covers {token_to_time(10)[0]:.2f}-{token_to_time(10)[1]:.2f} s")
mid = duration / 2
print(f"time {mid:.2f}s falls in audio token {time_to_token(mid)} "
      f"-> input_ids position {start + time_to_token(mid)}")


**Caveat worth keeping in mind.** This mapping is positional, not a claim
about receptive fields. The encoder runs 32 layers of *global* self-attention
before pooling, so every token has already seen the whole clip. Use this to
index and patch; don't use it alone to argue a token "encodes" a given moment.

## 5. Confirm against the real model (GPU)

Everything above is processor-only. This last section loads the 7B model to
check the actual tensor shapes coming out of the audio encoder.

In [ ]:
import torch
from transformers import Qwen2AudioForConditionalGeneration

model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.float16
)
model.eval()
print("loaded:", model.dtype, model.device)


In [ ]:
inputs, duration = encode(examples[0])
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

# Newer transformers moved the encoder under model.model.
audio_tower = getattr(model, "audio_tower", None) or model.model.audio_tower

# Cross-check our formula against the model's own length function.
valid_frames = int(inputs["feature_attention_mask"].sum())
_, official = audio_tower._get_feat_extract_output_lengths(torch.tensor([valid_frames]))
print(f"tokens: ours={n_audio_tokens(valid_frames)}  model's own fn={int(official[0])}")

# Audio encoder output, before it's projected into the LM.
with torch.no_grad():
    enc = audio_tower(inputs["input_features"].to(model.dtype)).last_hidden_state

print(f"\nclip duration        {duration:.2f} s")
print(f"encoder output       {tuple(enc.shape)}   (batch, frames, d_model)")
print(f"after pooling        ~{enc.shape[1] // 2} tokens -> projected to {config.text_config.hidden_size}")

# Hidden states at every layer: the LM's width is constant, length is what varies.
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)

print(f"\nLM layers            {len(out.hidden_states)}")
print(f"hidden state shape   {tuple(out.hidden_states[-1].shape)}   (batch, seq, hidden)")


## Next

We now know where the audio lives in the sequence and what each position
corresponds to in time. That's the addressing we need for attention analysis
and activation patching on the conflict examples.